# Yield Curve PCA Decomposition

In [1]:
from PIL import Image

import os
dir_path = "C:/Admin/CQF Course/Python Labs/Yield Curve Decomposition"

## Dimensionality Reduction

One of the main difficulties in today’s environment is being able to visualize data easily. There is too much information, too much news, and too much data. Dimensionality is the number of dimensions, features or input variables associated in a dataset and dimensionality reduction means reducing the number of features in a dataset.

Dimensionality reduction algorithms project high-dimensional data to a low-dimensional space while retaining as much of the variation as possible. There are two main approaches to dimensionality reduction.

The first one is known as linear projection which involves linearly projecting data from a high-dimensional space to a low-dimensional space. This includes techniques such as principal component analysis (PCA).
The second approach is known as manifold learning which is also referred to as nonlinear dimensionality reduction. This includes techniques such as Uniform manifold approximation and projection (UMAP).
Dimensionality reduction techniques help to address the curse of dimensionality.

## Principal Component

PCA is a linear dimensionality reduction techniqu where the algorithm finds a low-dimensional representation of the data while retaining as much of the variation as possible and help reduce the complexity.

The main concept behind the PCA is to consider the correlation among features. If the correlation is very high among a subset of the features, PCA will attempt to combine the highly correlated features and represent this data with a smaller number of linearly uncorrelated features. The algorithm keeps performing this correlation reduction, finding the directions of maximum variance in the original high-dimensional data and projecting them onto a smaller dimensional space. These newly derived components are known as principal components.

Investors often refer to movements in the yield curve in terms of three driving factors:

- Level
- Slope
- Curvature

PCA formalizes this viewpoint and allows us to evaluate when a segment of the yield curve has cheapened or richened beyond that prescribed by recent yield movements. The essence of PCA in the context of rates market is that most yield curve movements can be represented as a set of two to three independent driving factors – the principal components (PCs) – along with their relative weightings. And, with these components, it is possible to reconstruct the original features.

We'll apply PCA to the set of yield curves fitted using the HJM model as discussed during the lecture. The PCs are ordered so that the first PC is the most important in capturing variability in the yield curves, the second PC is next most important, and so on.

The most intuitive way of obtaining PCs is via eigenvalue decomposition of a covariance matrix. The covariance measures the central tendency and talks about deviation from the mean. Intuitively, PCs represent ways in which the forward rates making up a yield curve can deviate from their mean levels.

### Import Libraries : Exhaustive list, not all being used

In [2]:
# data manipuation
import pandas as pd
import numpy as np

# panda technical analysis 
import pandas_ta as ta

# data visualization
import matplotlib as mpl

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10,6)
plt.style.use ='seaborn'
plt.rcParams['font.size'] = 8

#-------------------------------------------------------------------------------------------------------------
# statistical visual representation
import seaborn as sns

import cufflinks as cf                                                        # new package - cufflinks
cf.set_config_file(offline=True, theme='pearl',dimensions=((1000,600)))       # cf.set_config() -- (w,h)

import plotly.express as px
px.defaults.width, px.defaults.height = 1000,600

#-------------------------------------------------------------------------------------------------------------
# Self organizing maps(SOM)
from minisom import MiniSom

#-------------------------------------------------------------------------------------------------------------
# K-means clustering(KMC)
from sklearn.cluster import KMeans

# scikit learn(sklearn) package is used primarily for machine learning
# always remember to perform pre-processing and model selcetion before using model implementation
import sklearn
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split,TimeSeriesSplit,cross_val_score,GridSearchCV,RandomizedSearchCV

from sklearn.pipeline import Pipeline

# sklearn evaluation metrics
from sklearn.metrics import precision_recall_curve, roc_curve,RocCurveDisplay,ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, f1_score, recall_score,precision_score,roc_auc_score,auc
from sklearn.metrics import classification_report, confusion_matrix

# Regression
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

# K-means clustering(KMC)
from sklearn.cluster import KMeans

# Support Vector Machine(SVM)
from sklearn.svm import SVC

# DTEM(Decision Trees and Ensemble Models) - Bagging - Random Forest
from sklearn.ensemble import RandomForestClassifier

# K nearest neighbours(KNN)
from sklearn.neighbors import KNeighborsClassifier

# Principal Component Analysis(PCA)
from sklearn.decomposition import PCA

#-------------------------------------------------------------------------------------------------------------
# DTEM(Decision Trees and Ensemble Models) - Boosting - Gradient Boosting(Extreme Gradient Boosting)
from xgboost import XGBClassifier,plot_importance,to_graphviz

#-------------------------------------------------------------------------------------------------------------
# statistical analysis & optimization
import scipy.stats as scs
import scipy.optimize as sco
from scipy.interpolate import interp1d
from scipy.integrate import simpson
from scipy.interpolate import CubicSpline

# statsmodels is a Python package that provides a complement to scipy for statistical computations
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

#-------------------------------------------------------------------------------------------------------------
# Nelson Siegel Svensson Model
from nelson_siegel_svensson import NelsonSiegelCurve, NelsonSiegelSvenssonCurve
from nelson_siegel_svensson.calibrate import calibrate_ns_ols, calibrate_nss_ols

#-------------------------------------------------------------------------------------------------------------
# SABR(Stochastic Alpha, Beta, Rho)
from pysabr import Hagan2002LognormalSABR
from pysabr import Hagan2002NormalSABR
from pysabr.helpers import year_frac_from_maturity_label

#-------------------------------------------------------------------------------------------------------------
# Shapley values, used in DTEM
import shap

#-------------------------------------------------------------------------------------------------------------
# Quantlib package
from QuantLib import *

#-------------------------------------------------------------------------------------------------------------
# Pyfolio package used for strategy backtesting
import pyfolio as pf

# to reload another python script 
from importlib import reload

# Ignore warnings - optional
import warnings
warnings.filterwarnings('ignore')

# data retrival
import yfinance as yf
 
# float display
pd.options.display.float_format = '{:0.4f}'.format

# operating system
import os

# Image package
from PIL import Image

# table formation
from tabulate import tabulate

# datetime handling
from datetime import datetime

import itertools

Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.


## PCA Implementation

### Read Data

In [3]:
data = pd.read_csv(os.path.join(dir_path, 'Fixed Income Lab - Curve Data and PCA/data/BLG daily 2002 to 2007 hjm-lecture.txt'),index_col=0,sep='\t')

Representation of a yield curve as 50 forward rates. As the yield curve evolves over time, each forward rate can change. It is understood that adjacent points on the yield curve do not move independently. PCA is a method for identifying the dominant ways in which various points on the yield curve move together.

PCA allows us to take a set of yield curves, process them using standard mathematical methods, and then define a reduced form model for the yield curve. This reduced form model retains only a small number of principal components (PCs) but can reproduce the vast majority of yield curves that the full structural model could. This reduced model has fewer sources of uncertainty (i.e. dimensions) than if the 50 points of the yield curve were modelled independently.

### Term Structure of Forward Rate

In [4]:
(data.iloc[0,:]).iplot(title = 'Representation of a Yield Curve in % terms')

### Derive Volatility

We'll now produce the volatility chart by taking the first difference (scaling) and calculating historical variance by each individual maturity.The drift of forward rate is fully determined by volatility of forward rate dynamics.


The volatility plot is of the averaged values, but we can see that different parts of the yield curve move differently. As you can see volatility is very significant, especially at the shorter end of the curve. This means that 1-year and 2-year rates seems to move up and down a lot as compared to other tenors.

It is never all up or all down and PCA help us figure out exactly what is going. Covariance of daily changes shows dependency of different rates. Principal components can be calculated by finding the eigenvalues and eigenvectors of this covariance matrix of below.

In [5]:
df_diff = data.diff(periods =-1)
df_diff.dropna(inplace=True)
df_vol= df_diff.std(axis = 0)*10000

df_vol.iplot(title='Volatility of daily UK government yields', xTitle='Tenor', yTitle='Volatility (bps)',color='cornflowerblue')

### Calculate Covariance

In [6]:
# since all these values are in % and therefore we need to divide by 10000 to show correct representation

df_cov = pd.DataFrame(np.cov(df_diff,rowvar=False)*252/10000)  # multiply with 252 to make it annualized
df_cov.head().style.format("{:.4%}")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50
0,0.0040%,0.0009%,0.0002%,-0.0001%,-0.0001%,-0.0000%,0.0001%,0.0001%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0002%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0000%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%,0.0001%
1,0.0009%,0.0063%,0.0055%,0.0041%,0.0035%,0.0033%,0.0031%,0.0029%,0.0028%,0.0027%,0.0026%,0.0025%,0.0024%,0.0022%,0.0021%,0.0020%,0.0019%,0.0018%,0.0017%,0.0016%,0.0015%,0.0014%,0.0013%,0.0012%,0.0011%,0.0011%,0.0010%,0.0009%,0.0009%,0.0008%,0.0008%,0.0008%,0.0007%,0.0008%,0.0008%,0.0008%,0.0008%,0.0008%,0.0008%,0.0009%,0.0009%,0.0009%,0.0010%,0.0010%,0.0010%,0.0011%,0.0012%,0.0012%,0.0012%,0.0013%,0.0013%
2,0.0002%,0.0055%,0.0082%,0.0077%,0.0068%,0.0061%,0.0056%,0.0052%,0.0048%,0.0045%,0.0042%,0.0040%,0.0038%,0.0036%,0.0035%,0.0033%,0.0032%,0.0031%,0.0029%,0.0028%,0.0027%,0.0026%,0.0025%,0.0023%,0.0022%,0.0021%,0.0020%,0.0020%,0.0019%,0.0018%,0.0018%,0.0017%,0.0017%,0.0017%,0.0017%,0.0017%,0.0017%,0.0017%,0.0017%,0.0018%,0.0018%,0.0018%,0.0019%,0.0019%,0.0020%,0.0021%,0.0021%,0.0022%,0.0022%,0.0023%,0.0024%
3,-0.0001%,0.0041%,0.0077%,0.0082%,0.0075%,0.0069%,0.0063%,0.0058%,0.0055%,0.0051%,0.0049%,0.0046%,0.0044%,0.0042%,0.0041%,0.0039%,0.0038%,0.0036%,0.0035%,0.0034%,0.0032%,0.0031%,0.0029%,0.0028%,0.0027%,0.0026%,0.0025%,0.0024%,0.0023%,0.0022%,0.0022%,0.0021%,0.0021%,0.0021%,0.0021%,0.0021%,0.0021%,0.0021%,0.0021%,0.0022%,0.0022%,0.0022%,0.0023%,0.0023%,0.0025%,0.0025%,0.0026%,0.0026%,0.0027%,0.0027%,0.0028%
4,-0.0001%,0.0035%,0.0068%,0.0075%,0.0072%,0.0067%,0.0063%,0.0059%,0.0056%,0.0054%,0.0051%,0.0049%,0.0047%,0.0046%,0.0044%,0.0043%,0.0041%,0.0039%,0.0038%,0.0036%,0.0035%,0.0033%,0.0031%,0.0030%,0.0029%,0.0027%,0.0026%,0.0025%,0.0024%,0.0023%,0.0023%,0.0022%,0.0022%,0.0022%,0.0022%,0.0022%,0.0022%,0.0022%,0.0022%,0.0023%,0.0023%,0.0024%,0.0025%,0.0025%,0.0026%,0.0026%,0.0027%,0.0028%,0.0028%,0.0029%,0.0030%


### Eigen Decomposition

In [7]:
eigenvalues, eigenvectors = np.linalg.eig(df_cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:,idx]

df_eigenval = pd.DataFrame({'EigenValue':eigenvalues})

### Explained Variance

In [8]:
df_eigenval['Explained Proportion'] = df_eigenval['EigenValue']/(df_eigenval['EigenValue'].sum())
df_eigenval.head()

,EigenValue,Explained Proportion
0,0.0020,0.7081
1,0.0005,0.1617
2,0.0002,0.0570
3,0.0001,0.0297
4,0.0001,0.0178


In [9]:
df_pca = pd.DataFrame(eigenvectors[:,:3], columns =['PC1','PC2','PC3'])

### Visualize PCs

In [10]:
df_pca.iplot(title='First Three Principal Components')

One of the key interpretations of PCA as applied to interest rates are the components of the yield curve. We can attribute the first three principal components to

- Parallel shifts in yield curve (shifts across the entire yield curve)
- Changes in short/long rates (steepening/flattening of the curve)
- Changes in curvature of the model (twists)

The first PC represents the situation that all forward rates in the yield curve move in the same direction but points around the 15 year term move more than points at the shorter or longer parts of the yield curve. This corresponds to a general rise (or fall) of all of the forward rates in the yield curve, but cannot be called a uniform or parallel shift. The impact of the first PC can be easily observed amongst the yield curves as it contributes more than 71% of the variability.

The second PC represents situations in which the short end of the yield curve moves up at the same time as the long end moves down, or vice versa. This is often described as a tilt in the yield curve, although in practice there is more subtle definition to the shape. This reflects the particular yield curves that were used for the analysis, as well as the structural model and calibration that were used to create them. In this excample, the influence of the second PC accounts for about 16.27% of the variability in the yield curves.

The third PC is further interpreted as a higher order buckling in which the short end and long end move up at the same time as a region of medium term rates move down, or vice versa. In this particular example, this type of movement is only responsible for about 5.75% of the variability.

Having identified the most important factors, we can use their functional form to predict the most likely evolution of the yeild curve. Thus, a simple linear regression is fitted for the shift factor as it simply moves the curve up and down. Second degree polynomial is fitted for the tilt factor and higher degree can approximate flexing. Thus, yield curve can be approximated by linear combination of first three loadings.

# PCA Implementation on UK Government Bond Rates

The purpose of applying PCA to financial markets is to explain the price changes of different assets through a smaller set of factors. This is achieved via the dimensionality reduction of the observations where we pick meaningful factors (among many) explaining the most of the price changes. We'll now apply the principal component analysis to UK government bond spot rates [1] 
from 0.5 years up to 10 years to maturity.

We'll adopt how two methods to decompose the yield curve: one using eigen decomposition as we above and another by applying direct functions from scikit-learn.

### Read data

In [11]:
data_uk = pd.read_excel(io= os.path.join(dir_path, 'Fixed Income Lab - Curve Data and PCA\data\GLC monthly_1970 to 2015.xlsx'),
                     sheet_name = '4. spot curve',index_col=0,header=3,skiprows=[4])
data_uk.dropna(inplace = True)

### Scale data(Standardization)

**Its always a good practice to standadarize data in PCA**

In [1]:
scaler = StandardScaler()                           # initialize the scaler - StandardScaler()   
scaler.fit(data_uk)                                 # scaler.fit() - Compute the mean and std to be used for later scaling
df = pd.DataFrame(scaler.transform(data_uk))        # Perform standardization by centering and scaling1 jm

NameError: name 'StandardScaler' is not defined

### Covariance Matrix

In [13]:
# here no need to annualize, already the case and since we have standardized data so no need to divide by 10000 
df_cov = pd.DataFrame(np.cov(df,rowvar=False))  
df_cov.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,1.0043,1.0017,0.9960,0.9886,0.9806,0.9723,0.9639,0.9556,0.9471,0.9385,...,0.6354,0.6282,0.6212,0.6145,0.6079,0.6016,0.5956,0.5897,0.5841,0.5786
1,1.0017,1.0043,1.0023,0.9974,0.9911,0.9840,0.9767,0.9691,0.9612,0.9531,...,0.6508,0.6435,0.6365,0.6298,0.6233,0.6170,0.6109,0.6051,0.5994,0.5940
2,0.9960,1.0023,1.0043,1.0028,0.9992,0.9943,0.9888,0.9827,0.9761,0.9691,...,0.6755,0.6683,0.6612,0.6544,0.6478,0.6415,0.6354,0.6295,0.6238,0.6183
3,0.9886,0.9974,1.0028,1.0043,1.0033,1.0005,0.9968,0.9922,0.9870,0.9813,...,0.6994,0.6921,0.6850,0.6782,0.6716,0.6652,0.6590,0.6531,0.6473,0.6418
4,0.9806,0.9911,0.9992,1.0033,1.0043,1.0035,1.0014,0.9983,0.9944,0.9898,...,0.7209,0.7136,0.7066,0.6997,0.6931,0.6867,0.6805,0.6745,0.6688,0.6632


### Eigen Decomposition,Explained Variance,Visualize PCs

In [14]:
eigenvalues, eigenvectors = np.linalg.eig(df_cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:,idx]
df_eigenval = pd.DataFrame({'EigenValue':eigenvalues})

In [15]:
df_eigenval['Explained Proportion'] = df_eigenval['EigenValue']/(df_eigenval['EigenValue'].sum())
df_eigenval.head()

,EigenValue,Explained Proportion
0,45.8355,0.9127
1,4.0762,0.0812
2,0.2630,0.0052
3,0.0362,0.0007
4,0.0042,0.0001


**we need to give negative sign for match**

In [16]:
# we need to give negative sign for match

df_pca = pd.DataFrame(eigenvectors[:,:2], columns =['PC1','PC2'])
df_pca = df_pca*(-1)

In [17]:
df_pca.iplot(title='First Three Principal Components')

## PCA Decomposition using Sklearn

### I prefer first one because it gives flexibility to modify data to find and top eigenvalues

### Model Pre-implementation & Model Implementation

In [18]:
# since we have already scaled data using StandardScaler() so Model pre-implementation is already covered

#---------------------------------------------------------------------------------------------------------------
# Initialize the model --> fit()
#---------------------------------------------------------------------------------------------------------------

pca = PCA()    
pca.fit(df_cov)                           

PCA()

### Model Attributes

In [19]:
pca.components_.T               # pca.components_: eigenvectors in row format, need to transpose to take it in columns
pca.explained_variance_         # pca.explained_variance_ : variance explained by each eigenvector OR EIGENVALUES

pca.explained_variance_ratio_   # eigen values proportion

array([8.51510080e-01, 1.48408624e-01, 7.78523785e-05, 2.92903899e-06,
       4.70149437e-07, 4.35828342e-08, 1.08258953e-09, 1.13538271e-11,
       1.29850030e-12, 5.73143570e-13, 1.02759944e-14, 3.42789261e-15,
       7.78730093e-16, 7.66508249e-18, 1.50054833e-18, 4.13524387e-19,
       6.46914062e-20, 1.26140496e-20, 1.62798769e-21, 8.79883267e-22,
       1.50722750e-22, 5.14748769e-23, 3.69491134e-23, 6.24443859e-24,
       1.42673858e-24, 1.06504911e-24, 1.75598843e-25, 5.66226125e-26,
       1.33608849e-26, 8.12647129e-27, 5.19468537e-27, 2.31824394e-27,
       1.16742333e-27, 4.01092019e-28, 3.03069561e-28, 1.71840238e-28,
       1.31002350e-28, 3.67953305e-29, 1.62407334e-29, 4.83005178e-30,
       2.48638915e-30, 1.89731117e-30, 1.10934212e-30, 3.22231874e-31,
       1.21917419e-31, 6.86212727e-32, 5.93445816e-32, 3.31553388e-32,
       3.28912017e-33, 2.18355356e-33])

### PCA Projection

In [20]:
df_pca_projec = df@df_pca
df_pca_projec.columns = ['PC1','PC2']

### PC1 - Yield

In [21]:
df_pca_projec.index = data_uk.index

level = pd.DataFrame({'10Y':data_uk.iloc[:,19],'PC1':df_pca_projec['PC1']})

In [22]:
level.iplot(title='PC1 : Yield',secondary_y='PC1')

### PC2 Slope

In [23]:
slope = data_uk.loc[:,['2Y','10Y']]
slope['Slope'] = slope['10Y']- slope['2Y']
slope['PC2'] = df_pca_projec['PC2']

In [24]:
slope.loc[:,['Slope','PC2']].iplot(title='PC2 : Slope',secondary_y='PC2')

## References

[Scikit-learn PCA Decomposition](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

[The Bank of England](https://www.bankofengland.co.uk/)